# 🚀 TAMC 2.0: Entrenamiento Intensivo en GPU (RTX 4090)
Este cuaderno contiene la estrategia mejorada con **Stop-Loss dinámico (ATR)**, indicadores de régimen de mercado (**ADX/Choppiness**) y un motor de entrenamiento correctedo con **GAE (Generalized Advantage Estimation)**.

In [ ]:
# 1. Instalación de dependencias (Ejecutar solo una vez)
%pip install yfinance torch pandas numpy plotly

In [ ]:
# 2. Imports y Configuración de GPU
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from dataclasses import dataclass
import os
import time
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
@dataclass
class StrategyConfig:
    ticker: str = "SOL-USD"
    period: str = "2y"      
    interval: str = "1h"    
    initial_capital: float = 10000.0
    n_features: int = 10    # Features base + ADX + Chop
    seq_length: int = 32    
    rl_gamma: float = 0.99
    rl_lambda: float = 0.95 # GAE parameter
    rl_learning_rate: float = 1e-4 
    ppo_clip_epsilon: float = 0.2
    ppo_epochs: int = 15    
    batch_size: int = 4096  # Ideal para los 24GB de la RTX 4090
    train_episodes: int = 200 
    hidden_dim: int = 256   
    commission_pct: float = 0.0005
    num_envs: int = 32      # Entornos en paralelo

In [ ]:
# 3. Procesamiento de Datos Mejorado
def robust_scaler_vec(series, window=100):
    rolling = series.rolling(window=window, min_periods=1)
    median = rolling.median()
    iqr = rolling.quantile(0.75) - rolling.quantile(0.25)
    return ((series - median) / (iqr + 1e-6)).clip(-5, 5)

def get_market_data(config):
    print(f"Descargando datos para {config.ticker}...")
    df = yf.download(config.ticker, period=config.period, interval=config.interval, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
    
    close = df['Close']
    delta = close.diff()
    df['RSI'] = 100 - (100 / (1 + (delta.where(delta > 0, 0).rolling(14).mean() / ((-delta.where(delta < 0, 0)).rolling(14).mean() + 1e-6))))
    df['MACD_Hist'] = close.ewm(span=12).mean() - close.ewm(span=26).mean()
    df['Log_Ret'] = np.log(close / close.shift(1))
    df['SMA200'] = close.rolling(200).mean()
    df['Dist_SMA200'] = (close - df['SMA200']) / (df['SMA200'] + 1e-6)
    tr = pd.concat([df['High']-df['Low'], (df['High']-close.shift()).abs(), (df['Low']-close.shift()).abs()], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(14).mean()
    df['ATR_Ratio'] = df['ATR'] / (tr.rolling(100).mean() + 1e-6)
    df['BB_Width'] = (close.rolling(20).std() * 4) / (close.rolling(20).mean() + 1e-6)
    df['VWAP'] = (close * df['Volume']).rolling(24).sum() / (df['Volume'].rolling(24).sum() + 1e-6)
    df['Dist_VWAP'] = (close - df['VWAP']) / (df['VWAP'] + 1e-6)
    
    # ADX & Choppiness
    plus_dm = np.where((df['High']-df['High'].shift(1)) > (df['Low'].shift(1)-df['Low']), np.maximum(df['High']-df['High'].shift(1), 0), 0)
    minus_dm = np.where((df['Low'].shift(1)-df['Low']) > (df['High']-df['High'].shift(1)), np.maximum(df['Low'].shift(1)-df['Low'], 0), 0)
    tr_s = tr.rolling(14).mean()
    plus_di = 100 * (pd.Series(plus_dm).rolling(14).mean() / (tr_s + 1e-6))
    minus_di = 100 * (pd.Series(minus_dm).rolling(14).mean() / (tr_s + 1e-6))
    df['ADX'] = (100 * (np.abs(plus_di - minus_di)/(plus_di + minus_di + 1e-6))).rolling(14).mean().values
    df['Choppiness'] = 100 * np.log10(tr.rolling(14).sum() / (df['High'].rolling(14).max() - df['Low'].rolling(14).min() + 1e-6)) / np.log10(14)

    cols = ['RSI', 'MACD_Hist', 'Dist_SMA200', 'Log_Ret', 'ATR_Ratio', 'BB_Width', 'Dist_VWAP', 'ADX', 'Choppiness']
    for col in cols: df[f'{col}_Norm'] = robust_scaler_vec(df[col])
    return df.iloc[205:].reset_index(drop=True)

In [ ]:
# 4. Modelo PPO-LSTM
class PPOActorCritic(nn.Module):
    def __init__(self, input_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc_in = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True, num_layers=2)
        self.actor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, action_dim), nn.Softmax(dim=-1))
        self.critic = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))
        
    def forward(self, x, hidden=None):
        x = self.fc_in(x)
        lstm_out, hidden_out = self.lstm(x, hidden)
        return self.actor(lstm_out[:, -1, :]), self.critic(lstm_out[:, -1, :]), hidden_out

In [ ]:
# 5. Motor de Entrenamiento Real
def compute_gae(rewards, values, next_value, dones, gamma, lmbda):
    advantages = torch.zeros_like(rewards).to(device)
    last_gae = 0
    for t in reversed(range(len(rewards))):
        next_val = next_value if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        advantages[t] = last_gae = delta + gamma * lmbda * (1 - dones[t]) * last_gae
    return advantages, advantages + values

def train():
    config = StrategyConfig()
    df_raw = get_market_data(config)
    cols_n = ['RSI_Norm', 'MACD_Hist_Norm', 'Dist_SMA200_Norm', 'Log_Ret_Norm', 'ATR_Ratio_Norm', 'BB_Width_Norm', 'Dist_VWAP_Norm', 'ADX_Norm', 'Choppiness_Norm']
    market_data = torch.FloatTensor(df_raw[cols_n].values).to(device)
    prices = torch.FloatTensor(df_raw['Close'].values).to(device)
    atrs = torch.FloatTensor(df_raw['ATR'].values).to(device)
    
    model = PPOActorCritic(config.n_features + 7, 5, config.hidden_dim).to(device)
    try: model = torch.compile(model)
    except: pass
    optimizer = optim.Adam(model.parameters(), lr=config.rl_learning_rate)
    
    print(f"\n>>> ENTRENANDO MODELO MEJORADO EN {config.num_envs} ENTORNOS PARALELOS")
    t0 = time.time()
    
    for ep in range(config.train_episodes):
        model.eval()
        cur_indices = torch.randint(config.seq_length, len(df_raw)-257, (config.num_envs,)).to(device)
        positions = torch.zeros(config.num_envs).to(device)
        entry_prices = torch.zeros(config.num_envs).to(device)
        balances = torch.ones(config.num_envs).to(device) * config.initial_capital
        
        mb_states, mb_actions, mb_rewards, mb_log_probs, mb_values, mb_dones = [], [], [], [], [], []
        
        for step in range(256):
            states_list = []
            for i in range(config.num_envs):
                idx = cur_indices[i].item()
                s = market_data[idx-config.seq_length:idx]
                meta = torch.zeros((config.seq_length, 7)).to(device)
                p_curr = prices[idx]
                p_entry = entry_prices[i]
                unrealized = (p_curr-p_entry)/p_entry if p_entry > 0 and positions[i] > 0 else (p_entry-p_curr)/p_entry if p_entry > 0 and positions[i] < 0 else 0
                meta[:, 0], meta[:, 1] = (1.0, 0.0) if positions[i] > 0 else (0.0, 1.0) if positions[i] < 0 else (0.0, 0.0)
                meta[:, 3] = unrealized * 10
                states_list.append(torch.cat([torch.zeros(config.seq_length, 1).to(device), s, meta], dim=1))
            
            states_t = torch.stack(states_list)
            with torch.no_grad(): probs, vals, _ = model(states_t)
            dist = Categorical(probs)
            actions = dist.sample()
            
            rewards = torch.zeros(config.num_envs).to(device)
            for i in range(config.num_envs):
                idx = cur_indices[i].item()
                target_pct = [0.0, 0.5, 1.0, -0.5, -1.0][actions[i].item()]
                p_curr = prices[idx]
                
                if target_pct != (positions[i]/balances[i] if balances[i] else 0):
                    balances[i] -= abs(target_pct*balances[i] - positions[i]) * config.commission_pct
                    positions[i], entry_prices[i] = balances[i] * target_pct, p_curr
                    rewards[i] -= 0.1
                
                pnl = (p_curr-entry_prices[i])/entry_prices[i] if positions[i] > 0 else (entry_prices[i]-p_curr)/entry_prices[i] if positions[i] < 0 else 0
                if pnl < -(2*atrs[idx]/p_curr) and positions[i] != 0:
                    balances[i] += pnl * abs(positions[i])
                    positions[i], rewards[i] = 0, rewards[i] - 1.0
                rewards[i] += pnl * 10
            
            mb_states.append(states_t); mb_actions.append(actions); mb_rewards.append(rewards)
            mb_log_probs.append(dist.log_prob(actions)); mb_values.append(vals.squeeze()); mb_dones.append(torch.zeros(config.num_envs).to(device))
            cur_indices += 1
            
        model.train()
        with torch.no_grad(): _, last_v, _ = model(states_t)
        advs, targets = compute_gae(torch.stack(mb_rewards), torch.stack(mb_values), last_v.squeeze(), torch.stack(mb_dones), config.rl_gamma, config.rl_lambda)
        
        b_states, b_actions, b_log_probs, b_advs, b_targets = torch.cat(mb_states), torch.cat(mb_actions), torch.cat(mb_log_probs), advs.view(-1), targets.view(-1)
        
        for _ in range(config.ppo_epochs):
            inds = torch.randperm(len(b_states))
            for i in range(0, len(b_states), config.batch_size):
                sb = inds[i:i+config.batch_size]
                curr_probs, curr_vals, _ = model(b_states[sb])
                curr_dist = Categorical(curr_probs)
                ratio = torch.exp(curr_dist.log_prob(b_actions[sb]) - b_log_probs[sb])
                loss = -torch.min(ratio * b_advs[sb], torch.clamp(ratio, 1-config.ppo_clip_epsilon, 1+config.ppo_clip_epsilon) * b_advs[sb]).mean() + 0.5 * (curr_vals.squeeze() - b_targets[sb]).pow(2).mean() - 0.01 * curr_dist.entropy().mean()
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        
        if (ep+1)%10 == 0: print(f"Ep {ep+1}/{config.train_episodes} | Avg Rew: {torch.stack(mb_rewards).mean():.4f} | Time: {time.time()-t0:.1f}s")

    torch.save(model.state_dict(), "tamc2_improved_4090.pth")
    print(f"\n>>> FINALIZADO. Modelo: 'tamc2_improved_4090.pth'")

train()